In [1]:
from requests import get, post
from sys import exit
from pprint import pprint
import pandas as pd
import time

In [2]:
HOST = "https://api.chartmetric.com"
TOKEN = "eVCH88EH3jriTh1ps2OpDtXeDMwlPPM7GY9y2J8jFkBY3RuLtx19DIXzC1sn0Ygm"

res = post(f"{HOST}/api/token", json={"refreshtoken": TOKEN})
if res.status_code != 200:
    print(f"ERROR: received a {res.status_code} instead of 200 from /api/token")
    exit(1)

access_token = res.json()["token"]

def Get(endpoint, params=None):
    res = get(
        f"{HOST}{endpoint}",
        headers={"Authorization": f"Bearer {access_token}"},
        params=params
    )
    if res.status_code != 200:
        print("ERROR:", res.status_code, res.text)
        return None
    return res.json()

In [3]:
genres = Get("/api/genre")
genres

{'obj': [{'id': 501120, 'name': 'pop'},
  {'id': 501121, 'name': 'hip-hop/rap'},
  {'id': 501122, 'name': 'rock'},
  {'id': 501123, 'name': 'metal'},
  {'id': 501124, 'name': 'electronic'},
  {'id': 501125, 'name': 'r&b/soul'},
  {'id': 501126, 'name': 'classical'},
  {'id': 501127, 'name': 'jazz'},
  {'id': 501128, 'name': 'blues'},
  {'id': 501129, 'name': 'devotional & spiritual'},
  {'id': 501130, 'name': 'folk'},
  {'id': 501131, 'name': 'country'},
  {'id': 501132, 'name': 'alternative'},
  {'id': 501133, 'name': 'eras'},
  {'id': 501134, 'name': 'ambient'},
  {'id': 501135, 'name': 'psychedelic'},
  {'id': 501136, 'name': 'spoken / comedy'},
  {'id': 501137, 'name': 'vocal'},
  {'id': 501138, 'name': 'singer/songwriter'},
  {'id': 501139, 'name': 'soundtrack'},
  {'id': 501140, 'name': 'instrumental'},
  {'id': 501141, 'name': 'holiday'},
  {'id': 501142, 'name': 'easy listening'},
  {'id': 501143, 'name': 'worldbeat'},
  {'id': 501144, 'name': 'industrial'},
  {'id': 501145, 'n

In [4]:
# STEP 1: USER INPUT

# Audience demographics
gender = "female"
min_gender_total = 50_000
age = ["18_24", "25_34"]
min_age_total = 10_000 # min absolute followers for each age group
audience_country = ["AR"]
min_audience_total = 100_000 # min absolute followers for each country
desired_interests = []

# Music filters
desired_genres = ["pop"]
genre_match_mode = "any" # "any" = at least one must match, "all" = all must match
desired_moods = []
mood_match_mode = "any"

# Artist filters 
artist_country = ["AR", "PE"] # one API call per country, results merged
metric = "sp_monthly_listeners"
min_pop = 0
max_pop = 1_000_000
limit = 200

In [5]:
# STEP 2: INITIAL ARTIST FETCH BY POPULARITY & COUNTRY

artists_raw = []

for code in artist_country:
    params = {
        "min": min_pop,
        "max": max_pop,
        "code2": code,
        "limit": limit
    }
    res = Get(f"/api/artist/{metric}/list", params=params)
    if res and "obj" in res:
        batch = res["obj"]["data"]
        artists_raw.extend(batch)
        print(f"  {code}: {len(batch)} artists found")
    else:
        print(f"  {code}: ERROR retrieving artists")


# Deduplicate by id in case an artist appears in multiple countries
seen = set()
artists_raw = [
    artist for artist in artists_raw
    if not (artist["chartmetric_artist_id"] in seen or seen.add(artist["chartmetric_artist_id"]))
]

print(f"\nStep 2: {len(artists_raw)} unique artists found across {artist_country}.\n")

  AR: 200 artists found
  PE: 200 artists found

Step 2: 400 unique artists found across ['AR', 'PE'].



In [6]:
artists_raw

[{'chartmetric_artist_id': 151039,
  'name': 'Esperanza de Vida',
  'code2': 'AR',
  'genres': 'ambient, beach, calm, chill, christian, christian & gospel, devoted, driving, free, fun, funky, grateful, heartbroken, heartwarming, inspirational, latin, latin christian, latin worship, lonely, longing, loving, noise, pop latino, rain, relaxed, sad, soothing, spa, spiritual, sun, tender, tranquil, uplifting, urban, worship, worshipping, yearning',
  'cpp_rank': 26115,
  'rank_eg': 25095,
  'rank_fb': 32036,
  'spotify_popularity': 51,
  'spotify_followers': 522001,
  'spotify_monthly_listeners': 999985,
  'spotify_listeners_to_followers_ratio': 1.915676,
  'spotify_followers_to_listeners_ratio': 0.522009,
  'deezer_fans': 78949,
  'facebook_likes': 0,
  'facebook_talks': 373,
  'twitter_followers': None,
  'twitter_retweets': None,
  'instagram_followers': 154430,
  'youtube_channel_views': 26874571,
  'youtube_subscribers': 89900,
  'youtube_daily_video_views': 25369,
  'youtube_monthly_vi

In [7]:
# STEP 3: FILTER BY GENRE AND MOOD

def matches_genre_mood(artist):
    tags = [t.strip().lower() for t in artist.get("genres", "").split(",")]

    mood_fn = all if mood_match_mode == "all" else any
    genre_fn = all if genre_match_mode == "all" else any

    mood_match = mood_fn(m.lower()  in tags for m in desired_moods)  if desired_moods  else True
    genre_match = genre_fn(g.lower() in tags for g in desired_genres) if desired_genres else True

    return mood_match and genre_match

artists_filtered = [artist for artist in artists_raw if matches_genre_mood(artist)]
print(f"Step 3: {len(artists_filtered)} artists remaining after genre/mood filter.\n")

Step 3: 93 artists remaining after genre/mood filter.



In [8]:
artists_filtered

[{'chartmetric_artist_id': 885462,
  'name': 'Maxi Espindola',
  'code2': 'AR',
  'genres': 'argentine telepop, latin ballad, latin pop, pop, pop urbano, teen pop',
  'cpp_rank': 25367,
  'rank_eg': 21788,
  'rank_fb': 50979,
  'spotify_popularity': 51,
  'spotify_followers': 20566,
  'spotify_monthly_listeners': 985427,
  'spotify_listeners_to_followers_ratio': 47.915346,
  'spotify_followers_to_listeners_ratio': 0.02087,
  'deezer_fans': 469,
  'facebook_likes': None,
  'facebook_talks': None,
  'twitter_followers': None,
  'twitter_retweets': None,
  'instagram_followers': 885381,
  'youtube_channel_views': None,
  'youtube_subscribers': None,
  'youtube_daily_video_views': None,
  'youtube_monthly_video_views': None,
  'wikipedia_views': None,
  'soundcloud_followers': 44,
  'bandsintown_followers': None,
  'tiktok_followers': None,
  'tiktok_likes': None,
  'sp_where_people_listen': [{'code2': 'ar',
    'listeners': 160396,
    'name': 'Buenos Aires'},
   {'code2': 'uy', 'listener

In [9]:
# STEP 4: FILTER BY AUDIENCE DEMOGRAPHICS

# Mapping age range
AGE_KEY_MAP = {
    "13_17": "ages_13_17",
    "18_24": "ages_18_24",
    "25_34": "ages_25_34",
    "35_44": "ages_35_44",
    "45_64": "ages_45_64",
    "65_": "ages_65_",
}

def get_audience_data(artist_id):
    return {
        "stat": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "stat"
        }),
        "demographic": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "demographic"
        }),
        "country": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "country"
        }) if audience_country else None,
        "interest": Get(f"/api/artist/{artist_id}/social-audience-stats", params={
            "domain": "instagram", "audienceType": "followers", "statsType": "interest"
        }) if desired_interests else None,
    }

In [10]:
def matches_demographics(artist):
    artist_id = artist.get("chartmetric_artist_id")
    artist_name = artist.get("name", "Unknown")

    all_data = get_audience_data(artist_id)

    # Total followers
    stat = all_data["stat"]
    if not stat or not stat.get("obj"):
        print(f"  ERROR: No stat data for {artist_name}")
        return False, {}

    total_followers = stat["obj"][0].get("followers", 0)
    if not total_followers:
        print(f"  ERROR: No follower count for {artist_name}")
        return False, {}
    print(f"  Follower count for {artist_name}: {total_followers}")

    # Gender Check
    demo = all_data["demographic"]
    if not demo or not demo.get("obj"):
        print(f"  ERROR: No demographic data for {artist_name}")
        return False, {}

    obj = demo["obj"][0]
    gender_total = round(obj.get(gender, 0) * total_followers)
    if gender_total < min_gender_total:
        return False, {"reason": f"{gender} total {gender_total:,} < min {min_gender_total:,}", "gender_total": gender_total, "total_followers": total_followers}
    print(f"  {gender} count for {artist_name}: {gender_total}")

    # Age Check
    age_totals = {}
    for bucket in age:
        age_key = AGE_KEY_MAP.get(bucket)
        if not age_key:
            print(f"  ERROR: Age bucket '{bucket}' not recognized.")
            return False, {}
        age_totals[bucket] = round(obj.get(age_key, 0) * total_followers)

    if not all(age_totals[b] >= min_age_total for b in age):
        failing = {b: age_totals[b] for b in age if age_totals[b] < min_age_total}
        return False, {"reason": f"age buckets below threshold: {failing}", "gender_total": gender_total, "age_totals": age_totals, "total_followers": total_followers}
    
    for bucket in age:
        print(f"  {bucket} count for {artist_name}: {age_totals[bucket]}")

    # Audience Country Check
    country_totals = {}
    if audience_country:
        country_obj = all_data["country"]
        if not country_obj or not country_obj.get("obj"):
            print(f"  ERROR: No country data for {artist_name}")
            return False, {}
        entries = country_obj["obj"]
        for code in audience_country:
            match = next((e for e in entries if e.get("code2", "").upper() == code.upper()), None)
            country_totals[code] = round(match.get("weights", 0) * total_followers) if match else 0
        if not all(country_totals[c] >= min_audience_total for c in audience_country):
            failing = {c: country_totals[c] for c in audience_country if country_totals[c] < min_audience_total}
            return False, {"reason": f"countries below threshold: {failing}", "gender_total": gender_total, "age_totals": age_totals, "country_totals": country_totals, "total_followers": total_followers}
        for code in audience_country:
            print(f"  {code} audience count for {artist_name}: {country_totals[code]:,}")

    # Interests Check
    if desired_interests:
        interest_obj = all_data["interest"]
        if not interest_obj or not interest_obj.get("obj"):
            print(f"  ERROR: No interest data for {artist_name}")
            return False, {}
        interest_names = [
            e.get("interest_name", "").lower()
            for e in interest_obj["obj"]
            if e.get("scale", 0) >= min_interest_pct
        ]
        if not any(i.lower() in interest_names for i in desired_interests):
            return False, {"reason": f"none of {desired_interests} found in interests: {interest_names}", "gender_total": gender_total, "age_totals": age_totals, "country_totals": country_totals, "total_followers": total_followers}
        matched_interests = [i for i in interest_names if i in [d.lower() for d in desired_interests]]
        print(f"  Interests matched for {artist_name}: {matched_interests}")

    return True, {
        "total_followers": total_followers,
        "gender_total":  gender_total,
        "age_totals":    age_totals,
        "country_totals":country_totals if audience_country else {},
    }

In [11]:
# Run filter
matched_artists = []

for artist in artists_filtered:
    name = artist.get("name", "Unknown")
    print(f"Checking: {name}...")

    is_match, demo = matches_demographics(artist)

    if is_match:
        matched_artists.append({**artist, "demographics": demo})
        print(f"  ✅ MATCH")
    else:
        if demo:
            print(f"  No match — {demo.get("reason", "unknown reason")}")
            print(f"     total_followers: {demo.get("total_followers", "N/A")}")

    time.sleep(0.3)

Checking: Maxi Espindola...
  Follower count for Maxi Espindola: 871533
  female count for Maxi Espindola: 760755
  18_24 count for Maxi Espindola: 431042
  25_34 count for Maxi Espindola: 318507
  ERROR: No country data for Maxi Espindola
Checking: Paula Prieto...
  Follower count for Paula Prieto: 24910
  No match — female total 14,346 < min 50,000
     total_followers: 24910
Checking: Chano...
  ERROR: No stat data for Chano
Checking: Phontana...
  ERROR: No stat data for Phontana
Checking: Elenco de Soy Luna...
  Follower count for Elenco de Soy Luna: 732
  No match — female total 659 < min 50,000
     total_followers: 732
Checking: Rudy La Scala...
  Follower count for Rudy La Scala: 60930
  No match — female total 44,191 < min 50,000
     total_followers: 60930
Checking: Martina Stoessel...
  Follower count for Martina Stoessel: 21632120
  female count for Martina Stoessel: 17910076
  18_24 count for Martina Stoessel: 9803655
  25_34 count for Martina Stoessel: 7907946
  AR audie

In [12]:
print(f"\n{len(matched_artists)} artists matched all criteria'\n")

for a in matched_artists:
    d = a["demographics"]
    print(f"🎵 {a['name']}")
    print(f"   Spotify listeners:  {a['spotify_monthly_listeners']:,}")
    print(f"   Total IG followers: {d['total_followers']:,}")
    print(f"   {gender.capitalize()} followers:   {d['gender_total']:,}")
    for bucket, total in d["age_totals"].items():
        print(f"   Age {bucket}:         {total:,}")
    for code, total in d["country_totals"].items():
        print(f"   Audience in {code}:   {total:,}")
    print(f"   Genres: {a['genres']}")
    print()


13 artists matched all criteria'

🎵 Martina Stoessel
   Spotify listeners:  879,152
   Total IG followers: 21,632,120
   Female followers:   17,910,076
   Age 18_24:         9,803,655
   Age 25_34:         7,907,946
   Audience in AR:   5,009,675
   Genres: abstract, affectionate, ambient, ambitious, anthemic, argentine telepop, beach, bittersweet, blue, bold, bonding, catchy, celebratory, cheerful, chill, connecting, cooking, cozy, cynical, dancy, dramatic, driving, emotional, empowering, encouraging, energetic, energizing, entertaining, family, feel-good, festive, focus, free, friendly, fun, funny, garden, guitar, happy, heartbroken, heartwarming, heavy, hopeful, hype, innocent, inspirational, intimate, jealous, latin, latino, latin pop, latin viral pop, lonely, loving, moody, mountains, moving, nature, noise, nostalgic, piano, pop, rain, raw, reflective, roadtrip, romantic, running, sad, sentimental, sleep, smooth, soft, soundtrack, soundtracks, spa, summer, sun, sweet, uniting, up